# Equity screening & valuation factors: walkthrough

Runs the pipeline and walks through its outputs: the ranked screen, backtest results, decile spreads,
P/E vs forward return, the out-of-sample model check, survivorship, earnings quality, a DCF with scenarios, the one-company research report and the validation report.

* `MODE = "demo"` runs offline on **synthetic** data with a planted value effect. Use it to check the
  install; the numbers say nothing about real stocks.
* `MODE = "live"` fetches real data (internet needed; API keys in `.env` for FMP or Alpha Vantage).
  Set `RUN_PIPELINE = False` to explore outputs from an earlier run without refetching.

In [ ]:
MODE = "demo"          # "demo" or "live"
RUN_PIPELINE = True

import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from eqscreen.config import load_config
from eqscreen.db import connect
from eqscreen.pipeline import run

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 20)
cfg = load_config(mode=MODE)
OUT = Path(cfg["project"]["output_dir"])
if RUN_PIPELINE:
    run(mode=MODE)
print(json.loads((OUT / "run_summary.json").read_text()))

## 1. Current screen
Top of the composite ranking, with the multiples an analyst would check first.

In [ ]:
screen = pd.read_csv(OUT / "screen_composite_latest.csv")
cols = ["rank", "ticker", "sector", "label", "composite", "pe", "pb", "ev_ebitda", "ev_ebit", "peg", "momentum_12_1"]
screen[cols].head(15).round(2)

In [ ]:
rule = pd.read_csv(OUT / "screen_rule_based_latest.csv")
rule[["rank", "ticker", "sector", "ev_ebit", "momentum_12_1", "label"]].head(10).round(2)

## 2. Backtest
Top-decile portfolios versus the equal-weight universe, net of costs.

In [ ]:
summary = pd.read_csv(OUT / "backtest_summary.csv")
net = summary[summary["series"] == "strategy_net"]
net[["strategy", "holding", "start", "end", "cagr", "sharpe", "excess_return_ann", "t_stat_excess",
     "max_drawdown", "avg_turnover_per_rebalance"]].round(3)

In [ ]:
monthly = pd.read_csv(OUT / "backtest_monthly.csv", parse_dates=["period_end"])
m = monthly[(monthly["strategy"] == "composite") & (monthly["holding"] == 1)].set_index("period_end")
ax = (1 + m[["net", "ew_universe", "benchmark"]].fillna(0)).cumprod().plot(logy=True, figsize=(9, 4))
ax.set_title("Growth of $1" + (" (synthetic demo data)" if MODE == "demo" else ""));

## 3. Does the score sort future returns?
Average forward return by composite decile, relative to the month's cross-section, and rank ICs.

In [ ]:
dec = pd.read_csv(OUT / "decile_returns.csv")
d = dec[(dec["score"] == "composite")].pivot(index="decile", columns="horizon_months", values="avg_excess_return")
d.plot(kind="bar", figsize=(9, 3.8), title="Excess forward return by composite decile (columns = horizon, months)");

In [ ]:
ic = pd.read_csv(OUT / "ic_summary.csv")
ic.pivot(index="score", columns="horizon_months", values="mean_ic").round(3)

## 4. P/E vs forward 12-month return
The factor panel lives in SQLite, so plain SQL works for ad hoc questions.

In [ ]:
conn = connect(cfg["project"]["db_path"])
pe = pd.read_sql_query("""
    SELECT month_end, ticker, sector, pe, ret_fwd_12m
    FROM factor_panel
    WHERE pe BETWEEN 0 AND 100 AND ret_fwd_12m IS NOT NULL
""", conn)
pe["pe_decile"] = pd.qcut(pe["pe"], 10, labels=False) + 1
pe.groupby("pe_decile")[["pe", "ret_fwd_12m"]].median().round(3)

In [ ]:
sector_val = pd.read_sql_query("""
    SELECT sector, COUNT(*) AS n, AVG(ev_ebit) AS mean_ev_ebit, AVG(pe) AS mean_pe
    FROM factor_panel
    WHERE month_end = (SELECT MAX(month_end) FROM factor_panel WHERE composite IS NOT NULL)
    GROUP BY sector ORDER BY mean_ev_ebit
""", conn)
sector_val.round(2)

## 5. Model check
Purged time-series CV: does a learned model beat the fixed-weight composite out of sample?

In [ ]:
status = json.loads((OUT / "model_status.json").read_text())
print(status)
if status["status"] == "ok":
    display(pd.read_csv(OUT / "model_cv.csv").round(3))
    display(pd.read_csv(OUT / "model_coefficients.csv").round(3))

## 6. Data quality and fetch log

In [ ]:
display(pd.read_csv(OUT / "quality_checks.csv")[["name", "status", "detail"]])
pd.read_sql_query("SELECT source, status, COUNT(*) AS n, MAX(fetched_at_utc) AS last_call FROM fetch_log GROUP BY source, status", conn)

## 7. Survivorship
Stocks are scored only while they were index members, but companies that left the index are mostly missing from free data. Treat backtest results as illustrative.

In [ ]:
summary = json.loads((OUT / "run_summary.json").read_text())
summary.get("survivorship")

## 8. One company: earnings quality
Pick the top-ranked non-financial stock that was an index member throughout.

In [ ]:
joiners = set(pd.read_sql_query("SELECT ticker FROM membership WHERE start_date IS NOT NULL", conn)["ticker"])
TICKER = screen[(screen["sector"] != "Financials") & ~screen["ticker"].isin(joiners)]["ticker"].iloc[0]

from eqscreen import earnings_quality as eq
from eqscreen.factors import load_panel
panel = load_panel(conn)
as_of = panel.loc[panel["composite"].notna(), "month_end"].max()
hist = eq.history(conn, cfg, TICKER, as_of)
print(TICKER, eq.summarize(hist, panel[panel["month_end"] == as_of], TICKER, cfg)["flags"])
hist.round(3)

## 9. DCF with scenarios and a reverse DCF
In **demo** mode the inputs below are illustrative values for a fictional company, only to show the mechanics. In **live** mode the notebook reads your own `config/dcf/<TICKER>.yaml` (create it with `python -m eqscreen.dcf --ticker X --init`) and never supplies assumptions for a real company.

In [ ]:
import tempfile, yaml
from eqscreen import dcf
from eqscreen.config import ROOT

if MODE == "demo":
    assume_path = Path(tempfile.mkdtemp()) / f"{TICKER}.yaml"
    dcf.init_assumptions(cfg, TICKER, None, assume_path)
    doc = yaml.safe_load(assume_path.read_text())
    doc.update({  # ILLUSTRATIVE inputs for a synthetic company, not market estimates
        "revenue_growth": [0.06, 0.05, 0.05, 0.04, 0.04], "ebit_margin": 0.20, "tax_rate": 0.21,
        "da_pct_revenue": 0.06, "capex_pct_revenue": 0.07, "nwc_pct_incremental_revenue": 0.10,
        "terminal_growth": 0.025, "risk_free_rate": 0.04, "equity_risk_premium": 0.05, "pre_tax_cost_of_debt": 0.055,
        "scenarios": {"bear": {"revenue_growth": [0.0] * 5, "ebit_margin": 0.15, "terminal_growth": 0.015},
                      "bull": {"revenue_growth": [0.09] * 5, "ebit_margin": 0.24, "terminal_growth": 0.03}}})
    assume_path.write_text(yaml.safe_dump(doc))
else:
    assume_path = ROOT / "config" / "dcf" / f"{TICKER}.yaml"

a = yaml.safe_load(assume_path.read_text())
base = dcf.load_base(cfg, TICKER, a.get("as_of"))
display(dcf.value_scenarios(base, a).round(3))
dcf.implied_assumptions(base, dcf.scenario_assumptions(a)["base"])

## 10. Research report (Markdown + HTML)
Marked DRAFT until the key-risks file is written (`python -m eqscreen.research --ticker X --init-risks`).

In [ ]:
from eqscreen.research import build_report
from eqscreen.render import write_html

text, meta = build_report(cfg, TICKER, assumptions_path=assume_path)
report_md = OUT / "research" / f"{TICKER}_notebook.md"
report_md.parent.mkdir(parents=True, exist_ok=True)
report_md.write_text(text, encoding="utf-8")
print(meta, write_html(report_md, f"{TICKER} research report"))
print(text[:1500])

## 11. Validation report
After a live run this documents filing availability, survivorship and real vs synthetic results. In demo mode both columns are synthetic.

In [ ]:
from eqscreen import validate
synth_cfg = load_config(mode="demo")
vpath = validate.build(cfg, synth_cfg)
pd.read_csv(vpath.parent / "real_vs_synthetic.csv")